# E2: decoding options of Whisper-large-v3-turbo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivannamor/multilingual-asr-lab/blob/main/notebooks/e2_decoding_colab.ipynb)

Experiment E2 of [RESULTS.md](https://github.com/ivannamor/multilingual-asr-lab/blob/main/RESULTS.md):
which decoding options make Whisper-large-v3-turbo faster without making it less accurate?

1. **Dev sweep.** The FLEURS dev sets (Italian, German, English) are transcribed with each
   configuration of [`experiments/E2/sweep.toml`](https://github.com/ivannamor/multilingual-asr-lab/blob/main/experiments/E2/sweep.toml):
   one decoding option changed at a time from the E1 setup.
2. **Selection.** A rule fixed in advance, in code (`asr_lab.sweep.select_configuration`), picks
   one configuration for all languages: the fastest one that is not significantly worse than the
   best one in any language. It reads only the dev runs.
3. **Test, once.** Only the selected configuration is run on the test sets; it is compared with
   the E1 run of large-v3-turbo. Nobody looks at the dev results between steps 1 and 3, so the
   test set cannot influence the choice.

This notebook only transcribes and applies the rule. Scores, comparisons and tables are recomputed
locally from the saved predictions, like every other result.

1. On Google Colab: *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. It takes
   about two hours; keep the tab open.
2. At the end the browser downloads `e2_runs.zip` (if it does not, download it from the *Files*
   panel). It contains `predictions.jsonl` and `run.json` of each run and the selection, no audio.
3. In a local clone, from the repository root: `python3 -m zipfile -e e2_runs.zip .`

If the session is interrupted, run all cells again: completed runs are skipped and an interrupted
run resumes where it stopped, as long as the Colab machine is the same. Timings are only
comparable within one machine, so a sweep resumed on another machine should be restarted.

## 1. Clone and install

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# Work inside a clone of the repository, so that every run.json records the git commit and the
# run directories end up where the experiment scripts expect them.
if Path("../pyproject.toml").exists():  # started from notebooks/ in a local clone
    os.chdir("..")
elif not Path("pyproject.toml").exists():  # on Colab
    if not Path("multilingual-asr-lab").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ivannamor/multilingual-asr-lab"],
            check=True,
        )
    os.chdir("multilingual-asr-lab")
if importlib.util.find_spec("asr_lab") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "."], check=True)
print(Path.cwd())

## 2. Settings

In [ ]:
SPEC_PATH = Path("experiments/E2/sweep.toml")  # model, languages and configurations
SELECTION_PATH = Path("experiments/E2/results/selection_dev.json")

### GPU setup

Same setup as the quickstart notebook: install the CUDA 12 libraries that CTranslate2 is built for
and load them explicitly. Unlike the quickstart, this notebook needs the GPU and stops if it is not
usable.

In [ ]:
import ctypes

import ctranslate2
import numpy as np

if ctranslate2.get_cuda_device_count() == 0:
    raise RuntimeError("No GPU: choose Runtime -> Change runtime type -> T4 GPU")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"],
    check=True,
)

In [ ]:
import nvidia.cublas
import nvidia.cudnn
from faster_whisper import WhisperModel

pending = [
    path
    for package in (nvidia.cublas, nvidia.cudnn)
    for path in sorted(Path(next(iter(package.__path__))).glob("lib/*.so*"))
]
for _ in range(5):  # the libraries depend on each other: retry the ones that failed
    failed = []
    for path in pending:
        try:
            ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL)
        except OSError:
            failed.append(path)
    pending = failed
    if not pending:
        break

check = WhisperModel("tiny", device="cuda", compute_type="float16")
list(check.transcribe(np.zeros(16_000, dtype=np.float32), language="it")[0])
print("GPU ready")

## 3. Data

The dev and test splits of each language, built exactly as on a local machine, so that the
manifests have the same SHA-256 and the predictions can be scored locally.

In [ ]:
from asr_lab.data.ingest import ingest_fleurs
from asr_lab.provenance import file_sha256
from asr_lab.sweep import load_spec

spec = load_spec(SPEC_PATH)
for lang in spec.languages:
    for split in ("dev", "test"):
        manifest = spec.manifest(lang, split)
        if not manifest.exists():
            ingest_fleurs(lang, split, manifest.parent)
        print(f"{manifest}  sha256 {file_sha256(manifest)}")
for config in spec.configs:
    print(f"{config.name:16} {config.options}")

## 4. Dev sweep

Every configuration on every language, one after the other (about an hour and a half). The model
is loaded once and shared by all configurations, and a short warm-up runs before the first timed
run, so that no configuration pays for initialising the GPU.

In [ ]:
from asr_lab.inference.faster_whisper import FasterWhisperModel
from asr_lab.sweep import run_sweep

model = FasterWhisperModel(spec.model_size, device="cuda")
run_sweep(spec, "dev", model=model)

## 5. Selection on the dev set

The rule of `asr_lab.sweep` (see its docstring), applied to the dev runs only. `*` marks the best
configuration of each language.

In [ ]:
from asr_lab.sweep import format_selection, save_selection, select_configuration

selection = select_configuration(spec, "dev")
save_selection(selection, SELECTION_PATH)
print(format_selection(selection))

## 6. Test set, once

Only the selected configuration (about 20 minutes).

In [ ]:
run_sweep(spec, "test", configs=[selection["selected"]], model=model)

## 7. Download the predictions

In [ ]:
import json
import zipfile

archive = Path("e2_runs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for run_dir in sorted(Path("experiments/runs").glob("e2-*")):
        run = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))
        print(f"{run_dir.name:45} RTF {run['rtf'] or float('nan'):.3f}")
        for name in ("predictions.jsonl", "run.json"):
            zipped.write(run_dir / name, (run_dir / name).as_posix())
    zipped.write(SELECTION_PATH, SELECTION_PATH.as_posix())
print(f"{archive.resolve()}: {archive.stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import files

    files.download(str(archive))
except ImportError:  # not on Colab
    pass

## 8. Exploratory: VAD on the test sets (optional)

Added **after** the dev results had been seen, so it is not part of the selection: on the dev
sets VAD was the most accurate configuration (German −0.31 points) but slightly slower, and the
rule selected `hst2`. This section measures whether the VAD gain holds on the test sets. It was
run in a separate Colab session after sections 1 to 3: `RUN_EXPLORATORY = True`, then only the
two cells below. Its speed is not comparable with the runs above (another machine).

In [ ]:
RUN_EXPLORATORY = False

if RUN_EXPLORATORY:
    import zipfile

    from asr_lab.inference.faster_whisper import FasterWhisperModel
    from asr_lab.sweep import run_sweep

    model = FasterWhisperModel(spec.model_size, device="cuda")
    exploratory_runs = run_sweep(spec, "test", configs=["vad"], model=model)

    archive = Path("e2_vad_test.zip")
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
        for run_dir in exploratory_runs:
            for name in ("predictions.jsonl", "run.json"):
                zipped.write(run_dir / name, (run_dir / name).as_posix())
    try:
        from google.colab import files

        files.download(str(archive))
    except ImportError:  # not on Colab
        pass